In [ ]:
# ── Colab setup (run this cell first!) ─────────────────────────────────────
import os, urllib.request

# Tell Keras to use JAX as its computing backend instead of TensorFlow.
# JAX should install cleanly on all platforms, including Apple Silicon Macs. (Fingers crossed)
os.environ.setdefault("KERAS_BACKEND", "jax")

# Enable interactive widgets (sliders, buttons) inside Google Colab.
# (and silently skip when running locally)
try:
    from google.colab import output as _out
    _out.enable_custom_widget_manager()
except ImportError:
    pass

# Download the utils and assignment files from GitHub into this Colab session
_RAW = "https://raw.githubusercontent.com/franzkeaw/NMRC-practicals-2026/main/"
for _f in ['utils.py']:
    if not os.path.exists(_f):  # skip if already downloaded in this session
        urllib.request.urlretrieve(_RAW + _f, _f)
# ─────────────────────────────────────────────────────────────────────────────

# Neurons as logic gates

## A little excursus on booleans and logic gates

Data with only two possible values (think of a neuron that can only produce output which is either 1 or 0) is handled by a branch of mathematics called _Boolean algebra_, where the values 1 and 0 are often replaced by the labels __true__ and __false__. Boolean algebra comes with a set of basic operations or functions, which take as input boolean values and produce boolean output. An example is the OR function, which given two boolean inputs, returns `True` if at least one of the inputs is `True`, or the exclusive or (XOR) function, which returns `True` only if exactly one of the two inputs is true. These operations are often summarized by so-called __truth tables__, like the one below (where $x$ and $y$ are the two inputs to the function).


| x | y | x AND y | x OR y | x XOR y |
|---|---|---------|--------|---------|
| 0 | 0 |0|0|0
| 1 | 0 |0|1|1
| 0 | 1 |0|1|1
| 1 | 1 |1|1|0

Most programming languages implement a boolean data type, which can be represented by 0 and 1 or by `True` and `False` labels. In Python, `True` and `False` are built-in constants, and we can ask Python to combine them with Boolean operations.

In [1]:
a = True
b = False
print(a or b)

True


We can also use the numeric values 1 and 0, as long as we tell Python that we mean boolean and not integer values. 

In [2]:
a = bool(1)
b = bool(0)
print(a or b)

True


In electronics, a __logic gate__ is a (conceptual) device which implements a boolean function. We can think of neurons as _biological logic gates_, in the sense that we can consider our modeled neuron as a system which receives binary inputs (e.g. spike or no spike from its presynaptic neurons) and outputs a binary signal (spike or no spike). In this tutorial we will explore how, depending on how the inputs are combined, a neuron can implement different boolean functions.

## Part 1 - A single neuron

We begin by writing a very simple Python class which we call `neuron`. This neuron has two methods, namely `__init__`, which is used to generate instances of this class, and `output`, which produces the neuron's output given the inputs. Below is also an example of how you can instantiate a neuron with all its parameters and call the `output` method (you don't _need_ to understand the code, but do take a look if you are interested).

In [1]:
import numpy as np
from utils import hardlim, hardlims
from ipywidgets import interact, fixed

class neuron():
    def __init__(self, bias, transfer_function, weights):
        self.bias = bias
        self.transfer_function = transfer_function
        self.weights = weights

    def output(self, inputs):
        # compute a weighted sum of the inputs
        x = np.sum([w * p for w, p in zip(inputs, self.weights)])
        # pass the sum through the activation function to produce the output
        return self.transfer_function(x - self.bias)


def neuron_for_widget(bias, transfer_function, weight_1, weight_2, input_1, input_2):
    my_neuron = neuron(bias=bias, transfer_function=transfer_function, weights=[weight_1, weight_2])
    out = my_neuron.output(inputs=[input_1, input_2])
    return bool(out)

# you can create an instance of the class neuron with a certain bias,
# transfer function, and with a list of weights that connect it to its input neurons
my_neuron = neuron(bias=0, transfer_function=hardlims, weights=[1, -1])

# then you can call the output method with a list of inputs to get the output
# of the neuron
out = my_neuron.output(inputs=[1, -1])
print('neuron output = ', out)

neuron output =  1


### Question 1
Write down the general equation that describes output $a$ as function of the inputs $p_i$ for some transfer function $f$. Then, simplify the equation for the case of a linear transfer function and two inputs $p_1$ and $p_2$.

### Answer 1
General equation: 
\begin{equation}
a = f(\sum_{i=1}^N w_i p_i - b)
\end{equation}

Simplification:
\begin{align}
a &= f(\sum_{i=1}^N w_i p_i - b) \\
&= f(w_1 p_1 + w_2 p_2 - b) \quad \mathrm{only \,two \, inputs} \\
&= w_1 p_1 + w_2 p_2 - b \quad \mathrm{linear \, transfer \, function}
\end{align}


[**Background.** This is the same neuron equation as in practical-1, now with several presynaptic inputs.
Each input $p_i$ is multiplied by its own synaptic weight $w_i$, the products are summed,
and the bias $b$ is subtracted --> a linear "vote counting" before the activation function $f$ is applied.
With a linear $f$ the neuron just computes a weighted sum minus a threshold offset.
That is precisely the form of a **linear classifier**: the set of inputs for which $a$ equals a given value is a
straight line (in 2-D), a plane (in 3-D), a hyperplane in general. Everything that follows in this notebook;
logic gates, the decision-boundary plot, and the impossibility of XOR, is about what this line/plane/hyperplane can and cannot separate.]

### Question 2
To get us started we consider a neuron with a `hardlim` transfer function. Run the cell below, then use the sliders to set the `bias=1`, `weight_1=1`, `weight_2=-1`. Then, try all possible four combinations of input values, and fill out a truth table like the one above, corresponding to the boolean operation performed by this neuron. Note that in the answer cell I already created the table for you, and listed the combinations of values of $p_1$ and $p_2$: you just have to edit the last column.

In [2]:
interact(neuron_for_widget, 
         bias=(0, 1, 0.5), transfer_function=fixed(hardlim), 
         weight_1=(-1, 1, 0.5), weight_2=(-1, 1, 0.5), input_1=(0, 1, 1), input_2=(0, 1, 1));

interactive(children=(FloatSlider(value=0.0, description='bias', max=1.0, step=0.5), FloatSlider(value=0.0, de…

### Answer 2
| p1 | p2 | neuron |
|---|---|---|
| 0 | 0 | 0 |
| 1 | 0 | 1 |
| 0 | 1 | 0 |
| 1 | 1 | 0 |


[**Background.** This question is a first taste of the neuron-as-logic-gate idea: with binary inputs and a hard-limit output, the neuron *is* a boolean function, and the truth table is the complete specification of that function (basically describing all possible inputs and output combinations). The parameters $(b=1, w_1=1, w_2=-1)$ implement "fire if $p_1$ is on and $p_2$ is off": the positive weight from $p_1$ pushes the neuron towards firing, the negative weight from $p_2$ vetoes it, and the bias 'requires' that the net input be at least zero. Note how the bias acts as a "how much evidence do I need"-knob. Also note the boundary subtlety: for input `(1,0)` the net input is *exactly* 0, and the output depends on the `hardlim` convention: under the convention we use here, the neuron fires.]


### Question 3
Next, run the cell below, and find values of the bias and the weights such that the output of our neuron becomes equivalent to the AND function shown previously.

In [4]:
interact(neuron_for_widget, 
         bias=(0, 1, 0.5), transfer_function=fixed(hardlim), 
         weight_1=(-1, 1, 0.5), weight_2=(-1, 1, 0.5), input_1=(0, 1, 1), input_2=(0, 1, 1));

interactive(children=(FloatSlider(value=0.0, description='bias', max=1.0, step=0.5), FloatSlider(value=0.0, de…

### Answer 3
Option 1: <br>
b = 1<br>
w1 = 0.5<br>
w2 = 0.5<br>

Option 2:<br>
b = 2<br>
w1 = 1<br>
w2 = 1<br>

(With $hardlim(0)=1$, an AND gate needs $0 < w_1 < b$, $0 < w_2 < b$ and $w_1 + w_2 \geq b$.)

[**Background.** AND is the "both inputs required"-detector. To implement it, the bias must demand *positive* evidence (so silence is the default), each individual input must be *insufficient* on its own (each weight smaller than the bias), and both together must *suffice* (the weights sum to at least the bias). This is the first example of a general recipe: to make a neuron compute a boolean function, write down the four inequalities that the truth table imposes on $w_1, w_2, b$ and check that they are simultaneously satisfiable. The recipe works for AND and OR but (as later practicals will show you) not for XOR, because XOR's inequalities are mutually contradictory.]



### Question 4
Now find parameter values such that the computation performed by the neuron corresponds to the OR function.

In [7]:
interact(neuron_for_widget, 
         bias=(0, 1, 0.5), transfer_function=fixed(hardlim), 
         weight_1=(-1, 1, 0.5), weight_2=(-1, 1, 0.5), input_1=(0, 1, 1), input_2=(0, 1, 1));

interactive(children=(FloatSlider(value=0.0, description='bias', max=1.0, step=0.5), FloatSlider(value=0.0, de…

### Answer 4

b = 0.5<br>
w1 = 1<br>
w2 = 1<br>

[**Background.** OR is the "at least one input required" detector, and it is *easier* to implement than AND in one respect: the individual weights must be *large enough* to overcome the bias on their own, whereas for AND they must be *too small* to do so. Compare the two solutions: OR = "any single vote is enough" ($w_i \geq b > 0$); AND = "no single vote is enough, but all together are" ($0 < w_i < b \leq w_1+w_2$).]



### Question 5
Can you also find parameter values which allow our neuron to compute the XOR operation?

### Answer 5
No

[**Background.** XOR ("exclusive or") is the smallest problem that is *not linearly separable*, and it historically played a big role: in 1969 Minsky and Papert's book *Perceptrons* showed that single-layer perceptrons cannot compute XOR, which contributed to the first "AI winter" (until it was realised that *multi-layer* networks can achieve this). The geometric intuition: the four input combinations are the corners of a square; a single neuron's decision boundary is a straight line; you can draw a line that puts `(0,0)` on one side and the rest on the other (OR), or `(1,1)` alone on one side (NAND), but no line puts exactly the two *diagonal* corners `(1,0)` and `(0,1)` on the same side while excluding the other diagonal. Part 2 of this notebook shows the fix: use two neurons (two lines) and combine their outputs with a third.]


### Question 6
Can you show mathematically why this is not possible? Hint: write out the equations for all combinations of input values (and fill in 0 for the bias).

### Answer 6
The neuron output is defined as:
$s = w_1 p_1 + w_2 p_2 - b$, $a = \mathrm{hardlim}(s)$

For the neuron to apply XOR, it should follow these constraints (as in table on top of this page):

|-| p1 | p2 | a |
|-|---|---|---------|
|(1)| 0 | 0 | 0
|(2)| 1 | 0 | 1
|(3)| 0 | 1 | 1
|(4)| 1 | 1 | 0

Writing $s(p_1, p_2) = w_1 p_1 + w_2 p_2 - b$, the four rows impose (with $\mathrm{hardlim}(0)=1$, i.e. the neuron fires if and only if $s \geq 0$):

(1) $s(0,0) < 0$: $-b < 0 \Rightarrow b > 0$

(2) $s(1,0) \geq 0$: $w_1 \geq b$

(3) $s(0,1) \geq 0$: $w_2 \geq b$

(4) $s(1,1) < 0$: $w_1 + w_2 < b$

Adding (2) and (3) gives $w_1 + w_2 \geq 2b$. Since $b > 0$ from (1), $2b > b$, hence $w_1 + w_2 > b$, which contradicts (4), $w_1 + w_2 < b$.

Thus the system cannot be solved: no single neuron can compute XOR.


[**Background.** This proof is a miniature version of a fundamental theorem: a linear threshold unit can only compute **linearly separable** boolean functions.]



Another way to think about what the neuron is doing is to plot each combination of input values on a 2-D plane. Then, the parameters of our neuron determine a line on this 2-D plane that will separate points in input space for which the neuron becomes active (responds with a 1) from points where the neuron remains inactive (responds with a 0). This line is known as the _decision boundary_. 

### Question 7
Derive the equation for the line which represents the decision boundary, through expressing p_2 as function of p_1. (this will allow you to visualize the boundary)

### Answer 7

Starting with:

$a = w_1 p_1 + w_2 p_2 - b $

Since the transfer function is a hardlim, it will be activated from zero and as such:

$0 = w_1 p_1 + w_2 p_2 - b $

$b = w_1 p_1 + w_2 p_2 $

$b-w_1 p_1 = w_2 p_2 $

$(b-w_1 p_1)/w_2 = p_2 $

[**Background.** The decision boundary is the set of inputs for which the neuron is exactly at threshold: the dividing line between "fires" and "doesn't fire". Because the pre-activation is linear in the inputs, that set is always a straight line in 2-D (a hyperplane in general): all points on one side share the same output. The weights determine the line's *orientation* (via the ratio $w_1/w_2$) and the bias its *position* (via $b/w_2$). This geometric picture is useful because it turns parameter search into (intuitive) geometry: implementing AND means placing the line so that only `(1,1)` is on the "on" side; OR means placing it so that only `(0,0)` is on the "off" side.]



Let's write up a function which will allow us to visualize the decision boundary.

In [2]:
import matplotlib.pyplot as plt

def plot_decision_boundary(weight_1, weight_2, bias):
    inputs = np.array([[0, 0],
                       [1, 1],
                       [0, 1],
                       [1, 0]])
    
    weights = [weight_1, weight_2]
    my_neuron = neuron(bias=bias, transfer_function=hardlim, weights=weights)

    f, ax = plt.subplots(1, 1, figsize=[6, 6])
    
    plot_line = lambda x : (bias-weights[0]*x)/ (weights[1] + 10e-5)
    xx, yy = np.arange(-2, 2, step=0.001), np.arange(-2, 2, step=0.001)

    ax.plot(xx, plot_line(xx), c='k')
    for i in range(inputs.shape[0]):
        if my_neuron.output(inputs[i,:])>0:
            ax.scatter(inputs[i, 0], inputs[i, 1], facecolors='k', edgecolors='k', s=104, linewidths=2)
        else:
            ax.scatter(inputs[i, 0], inputs[i, 1], facecolors='none', edgecolors='k', s=104, linewidths=2)
    ax.set_xlim([-1, 2])
    ax.set_ylim([-1, 2])
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_title('activation function: hardlim')
    plt.show()

Run the cell below, and using the sliders, explore the effect of the weights and of the bias on the decision boundary. Filled dots indicate that the neuron's output is 1 (true), and 0 (false) for the empty dots. 

In [3]:
interact(plot_decision_boundary, weight_1=(-1, 1, 0.1), weight_2=(-1, 1, 0.1), bias=(-1, 1, 0.05));

interactive(children=(FloatSlider(value=0.0, description='weight_1', max=1.0, min=-1.0), FloatSlider(value=0.0…

### Question 8
Set up the weights and bias you discovered in question 4 which implement the OR function. Do you get the expected filled/empty dots? 

### Answer 8
Yes.

[**Background.** This is the payoff of the geometric view: the truth table and the boundary plot are two representations of the same function, and checking one against the other is a quick sanity check of the parameters. Filled dots are the inputs where the linear score reaches the threshold; the line is where it equals the threshold. Notice that the boundary does not need to pass *between* the points with equal margin, because any line that separates the classes implements the same boolean function, which is why there are infinitely many correct parameter sets. This non-uniqueness will come back in the perception notebook later, where networks *learns* one of these solutions automatically, and different runs can find different solutions.]


### Question 9
What would the decision boundary have to look like for the XOR operation to be carried out? Give a graphical explanation of why XOR cannot be solved by a single linear classifier. 

### Answer 9

Curved decision boundary necessary. Not linearly separable.

[**Background.** "Linearly separable" means: there exists a straight line (hyperplane) putting all "on" points on one side and all "off" points on the other. AND, OR, NAND, NOR are all linearly separable; XOR (and XNOR) are the smallest non-separable boolean functions. The practical consequence, taken up in Part 2: to compute non-linearly-separable functions you need to *combine* linear boundaries: several neurons, each contributing one line, whose outputs are themselves combined. That is precisely what a hidden layer does in neural networks.]


## Part 2 - Extending to more layers

For now we have focused our attention on mechanisms which pertain to the functioning of a single neuron. But there must be a reason why the brain contains so many neurons! Intuitively, having more neurons should allow us to somehow do more complicated things. Although it is certainly not the only factor, the number of neurons tends to be larger in more evolved and cognitively capable species, as you can see [here](https://en.wikipedia.org/wiki/List_of_animals_by_number_of_neurons). In this part of the assignment we are going to show in a very simple yet very concrete example how using more neurons allows us to do more sophisticated computations. 

If you have completed Part 1 of the assignment it should be clear that the XOR problem (associating a response equal to 1 only when either one of the inputs is 1, and 0 otherwise) is just too much for a single neuron. At this point, you may understandably be disappointed in neurons, and wonder how the brain ever does anything computationally interesting. But don't lose faith: __the strength is in the union of many computationally simple elements__.

### Question 1
Consider again the decision boundary plot. Suppose now you can draw two straight lines instead of one. Can you now draw them in such a way as to compute the XOR function?

### Answer 1
Yes

[**Background.** XOR needs *two* boundaries, and two boundaries are available if we use two neurons. The first layer's neurons each compute a linearly separable function (OR and NAND, or two "single-input detectors"); the second layer combines their outputs with another linear threshold (an AND or an OR). The key insight is *hierarchical composition*: a network of simple linear-threshold units can compute functions that no single unit can, because the units can be arranged in layers where each layer's boundary operates in the *output space of the previous layer*.]


It seems like with two neurons we could be getting somewhere! But how do we proceed? The idea is that we can give the same input to two neurons, which can then process it independently (with different weights and biases) and give us back two outputs. Then we can have a neuron which takes as input the output of the first two neurons and gives us the final output. So, while the first and second neuron (the first layer) compute a logic function on the input, the neuron in the second layer (neuron 3) computes a logic function on the output of the first layer. This is summarized in the figure below: on the left we have the one neuron scenario, and on the right the two-layer architecture.

![title](https://raw.githubusercontent.com/franzkeaw/NMRC-practicals-2026/main/images/lg_1.svg)

### Question 2
In question 1 you were hopefully able to draw two lines that isolate (in the intersection of their half-planes) the points at $(1, 0)$ and $(0, 1)$. Translate this into truth table entries by filling in the __Output neuron 1__ and __Output neuron 2__ below.

### Answer 2
[Edit the entries in the table for neuron 1 and neuron 2]

There are two solutions. The first is to have one neuron compute an OR-function and the other a NOT-AND operation. The final computation by the second layer is an AND operation:

| Input 1 | Input 2 | Output neuron 1 | Output neuron 2 | Output neuron 3 |
|---------|---------|-----------------|-----------------|-----------------|
|    0    |    0    |      0          |        1        |        0        |
|    0    |    1    |      1          |        1        |        1        |
|    1    |    0    |      1          |        1        |        1        |
|    1    |    1    |      1          |        0        |        0        |

The second solution is to have one neuron is only activated by Input 1 (negative weight input 2) and the other only by Input 2 (negative weight input 1). The third neuron can perform an OR operation:

| Input 1 | Input 2 | Output neuron 1 | Output neuron 2 | Output neuron 3 |
|---------|---------|-----------------|-----------------|-----------------|
|    0    |    0    |      0          |        0        |        0        |
|    0    |    1    |      0          |        1        |        1        |
|    1    |    0    |      1          |        0        |        1        |
|    1    |    1    |      0          |        0        |        0        |


[**Background.** Each solution corresponds to one of the two ways of drawing two lines around the diagonal corners. Solution 1 draws a *band*: the OR line cuts off the origin corner, the NAND line cuts off the `(1,1)` corner, and XOR is the band between them --> the output neuron then just checks "inside the band" (AND of "above line 1" and "below line 2"). Solution 2 draws two *strips* that each isolate one "on" corner (a vertical strip around `p1=1` vetoed by `p2`, and vice versa), and the output neuron checks "in either strip" (OR). Both are correct; they differ in which intermediate functions the hidden neurons compute. Recognising both is useful because it shows that "the" solution to a network-design problem is often not unique (a fact that also explains why trained networks in later practicals find different weights on different runs).]



### Question 3
What logic function allows you to combine the outputs of neuron 1 and neuron 2 to obtain the desired XOR at the output of neuron 3?

### Answer 3
In the first case an AND-operation, in the second example (see above) the OR function.

[**Background.** The output neuron is itself just a logic gate operating not on the raw inputs but on the *features computed by the hidden layer*. This is the essence of what a hidden layer does: it **transforms the input into a new representation** in which the problem becomes linearly separable (or, here, reducible to a single gate). In solution 1 the hidden layer's representation is "how many inputs are on?" (0, 1, or 2) encoded as (OR, NAND); XOR is then "exactly 1", which is an AND of the two. In solution 2 the representation is "which single input is on?", and XOR is "one of them", an OR.]


### Question 4

We are now going to implement this network using the `neuron` object introduced at the beginning of this assignment. In the code below, we generate three neurons, and use them to compute the chained logic operation which allows us to compute the XOR function on our inputs. The output of network is computed for all input combinations. We then print out a truth table where the three columns correspond to input 1, input 2, and the final output (the output of the third neuron).

Find values for the weights and biases of the three neurons in our small network such that the output of the third neuron gives the correct XOR function on the inputs. Edit the code below by inserting the correct parameters, and run it to make sure that you get the desired output. You can use the decision boundary interactive plot from before to determine the parameters of each neuron. 

At the beginning, everything is initialized to 0: if you run the cell once you will see that you get a truth table printed out with only 1s in the output. Can you explain why?

In [5]:
# we construct our three neurons
neuron1 = neuron(bias=0, transfer_function=hardlim, weights=[0, 0])
neuron2 = neuron(bias=0, transfer_function=hardlim, weights=[0, 0])
neuron3 = neuron(bias=0, transfer_function=hardlim, weights=[0, 0])

# we make an array with all possible input combinations
inputs = np.array([[0, 0],
                   [0, 1],
                   [1, 0],
                   [1, 1]])

# for every input, we feed it to neuron 1 and neuron 2, and record their output.
# then, we feed the output of the first layer (neurons 1 and 2) to the third neuron,
# to get our final output.
for inpt in inputs:
    out1 = neuron1.output(inpt)
    out2 = neuron2.output(inpt)
    out3 = neuron3.output([out1, out2]) 
    print("%s | %s | %s " %(inpt[0], inpt[1], out3)) # print the results. the last column is the output of neuron 3

0 | 0 | 1 
0 | 1 | 1 
1 | 0 | 1 
1 | 1 | 1 


### Answer 4

One possible solution (fill these parameters into the code cell above and run it):

```python
neuron1 = neuron(bias=0.5, transfer_function=hardlim, weights=[1, 1])     # OR
neuron2 = neuron(bias=-1.5, transfer_function=hardlim, weights=[-1, -1]) # NOT AND
neuron3 = neuron(bias=1.5, transfer_function=hardlim, weights=[1, 1])     # AND
```

Running the cell then prints the XOR truth table (`0, 1, 1, 0` in the last column). The intermediate outputs of neurons 1 and 2 are listed in the first table of Answer 2 above; a second solution is worked out in Answers 5a-c below.

The all-1s output you get with the zero initialization comes from `hardlim(0) = 1`: with all weights and biases equal to 0, every neuron receives $s = 0$ and therefore fires, so the output of neuron 3 is 1 for every input.

[**Background.** To design the network, solve each neuron separately using the Q3/Q4 recipes, treating the hidden neurons' outputs as the inputs. Neuron 1 must be OR of $(p_1, p_2)$: $b>0$, $w_i \geq b$ → $(0.5; 1, 1)$. Neuron 2 must be NAND: fire everywhere except `(1,1)` --> negate the weights and set the bias so that only the `(1,1)` score falls below threshold: $w=[-1,-1]$, $b=-1.5$ (scores: 1.5, 0.5, 0.5, −0.5). Neuron 3 must be AND of the hidden outputs: by the Q3 conditions, $0 < w < b_3$ and $2w \geq b_3$ → with $w=1$: $1 < b_3 \leq 2$, e.g. $b_3 = 1.5$.]



### Question 5

It really does seem the strength is in the union of many computationally simple elements. Even to the extent that there is not just one solution to this problem but two. Given that you have found one solution by which the output of two intermediate neurons can be combined in a third neuron to solve the XOR problem, what is the second solution?

Go over the previous questions and answer for this other scenario:

What are truth table entries for __Output neuron 1__ and __Output neuron 2__ and __Output neuron 3__?

What logic function allows you to combine the outputs of neuron 1 and neuron 2 to obtain the desired XOR at the output of neuron 3?

Find values for the weights and biases of the three neurons in our small network such that the output of the third neuron gives the correct XOR function on the inputs. Edit the code below by inserting the correct parameters, and run it to make sure that you get the desired output. (Again: You can use the decision boundary interactive plot from before to determine the parameters of each neuron. )


### Question 5a
[Edit the entries in the table for neuron 1 and neuron 2]

| Input 1 | Input 2 | Output neuron 1 | Output neuron 2 | Output neuron 3 |
|---------|---------|-----------------|-----------------|-----------------|
|    0    |    0    |      ?          |        ?        |        0        | 
|    0    |    1    |      ?          |        ?        |        1        |
|    1    |    0    |      ?          |        ?        |        1        |
|    1    |    1    |      ?          |        ?        |        0        |

### Question 5b
What logic function allows you to combine the outputs of neuron 1 and neuron 2 to obtain the desired XOR at the output of neuron 3?

### Question 5c
Find values for the weights and biases of the three neurons for the second solution:

### Answer 5a
Truth table for the second solution (neuron 1 = $p_1$ AND NOT $p_2$, neuron 2 = NOT $p_1$ AND $p_2$):

| Input 1 | Input 2 | Output neuron 1 | Output neuron 2 | Output neuron 3 |
|---------|---------|-----------------|-----------------|-----------------|
|    0    |    0    |      0          |        0        |        0        |
|    0    |    1    |      0          |        1        |        1        |
|    1    |    0    |      1          |        0        |        1        |
|    1    |    1    |      0          |        0        |        0        |

### Answer 5b
The outputs of neuron 1 and neuron 2 are combined with an **OR** function: neuron 3 fires whenever either hidden neuron fires, which gives exactly the XOR pattern (0, 1, 1, 0).

### Answer 5c
One possibility (verify by running the code cell below with the correct values entered):

```python
neuron1 = neuron(bias=0.5, transfer_function=hardlim, weights=[1, -1])   # p1 AND NOT p2
neuron2 = neuron(bias=0.5, transfer_function=hardlim, weights=[-1, 1])   # NOT p1 AND p2
neuron3 = neuron(bias=0.5, transfer_function=hardlim, weights=[1, 1])    # OR
```

Each hidden neuron fires for exactly one of the two 'on' corners (positive weight from its preferred input, negative weight vetoing the other input, bias 0.5 in between), and the OR neuron fires if either fires, giving the output `0, 1, 1, 0` = XOR.

In [6]:
# we construct our three neurons
neuron1 = neuron(bias=0, transfer_function=hardlim, weights=[0, 0])
neuron2 = neuron(bias=0, transfer_function=hardlim, weights=[0, 0])
neuron3 = neuron(bias=0, transfer_function=hardlim, weights=[0, 0])

# we make an array with all possible input combinations
inputs = np.array([[0, 0],
                   [0, 1],
                   [1, 0],
                   [1, 1]])

# for every input, we feed it to neuron 1 and neuron 2, and record their output.
# then, we feed the output of the first layer (neurons 1 and 2) to the third neuron,
# to get our final output.
for inpt in inputs:
    out1 = neuron1.output(inpt)
    out2 = neuron2.output(inpt)
    out3 = neuron3.output([out1, out2]) 
    print("%s | %s | %s " %(inpt[0], inpt[1], out3)) # print the results. the last column is the output of neuron 3

0 | 0 | 1 
0 | 1 | 1 
1 | 0 | 1 
1 | 1 | 1 
